In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from students.degtyarev.lesson2 import Exercise


def prepare_iris_data():
    features, labels = load_iris(return_X_y=True)

    is_first_class = (labels == 0).astype(int)

    x_train_val, x_test, y_train_val, y_test = train_test_split(
        features, is_first_class, test_size=0.2, random_state=42, stratify=is_first_class
    )
    x_train, x_val, y_train, y_val = train_test_split(
        x_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
    )

    avg = x_train.mean(axis=0)
    deviation = x_train.std(axis=0)

    return {
        "train": ((x_train - avg) / deviation, y_train),
        "val": ((x_val - avg) / deviation, y_val),
        "test": ((x_test - avg) / deviation, y_test),
    }


def run_hyperparameter_search(data):
    x_train, y_train = data["train"]
    x_val, y_val = data["val"]

    rates = [0.005, 0.01, 0.05, 0.1, 0.3, 0.7, 1.0]
    sizes = [4, 8, 16, 32, 64, None]
    iterations = 40

    performance_log = []

    print(f"{'Learning Rate':<15} | {'Batch':<8} | {'F1 Score':<10} | {'Reliability':<10}")
    print("-" * 55)

    for rate in rates:
        for size in sizes:
            trial_scores = []
            for attempt in range(1, 6):
                model = Exercise.create_logistic_model(
                    num_features=x_train.shape[1], rng=np.random.default_rng(attempt)
                )
                Exercise.fit(model, x_train, y_train, lr=rate, n_iter=iterations, batch_size=size)
                trial_scores.append(model.f1_score(x_val, y_val))

            avg_f1 = np.mean(trial_scores)
            reliability = avg_f1 - np.std(trial_scores)

            performance_log.append({"lr": rate, "bs": size, "score": avg_f1, "rel": reliability})

            display_bs = str(size) if size else "Full"
            print(f"{rate:<15} | {display_bs:<8} | {avg_f1:<10.4f} | {reliability:<10.4f}")

    return max(performance_log, key=lambda item: item["rel"])


dataset = prepare_iris_data()
best_setup = run_hyperparameter_search(dataset)

print("\n" + "Результаты оптимизации".center(40, "="))
print(f"Оптимальный темп обучения (LR): {best_setup['lr']}")
print(f"Оптимальный размер батча:      {best_setup['bs']}")
print(f"Показатель надежности:         {best_setup['rel']:.4f}")
print("=" * 40)

x_train, y_train = dataset["train"]
x_test, y_test = dataset["test"]

final_model = Exercise.create_logistic_model(num_features=x_train.shape[1])
Exercise.fit(final_model, x_train, y_train, lr=best_setup["lr"], n_iter=100, batch_size=best_setup["bs"])

print("\nИтоговые метрики на независимой тестовой выборке:")
print(f"-> Точность (Accuracy): {final_model.accuracy(x_test, y_test):.4f}")
print(f"-> F1-мера:             {final_model.f1_score(x_test, y_test):.4f}")
print(f"-> AUROC:               {final_model.auroc(x_test, y_test):.4f}")